## Step 1 Data preparation 

In [ ]:
%pip install tensorflow numpy matplotlib seaborn

In [2]:
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.utils import to_categorical
import tensorflow as tf

# ============================================================
# CONFIGURATION
# ============================================================

ENGINEERED_DIR = r"D:\uni\Intern-1-Project\ksl\dataset\Landmark\New_data-landmark\landmarks_30frames_engineered"
SEQUENCE_LENGTH = 30
FEATURES_PER_FRAME = 686  # Your engineered features

# ============================================================
# LOAD AND PREPARE DATA
# ============================================================

def load_data(base_dir):
    """
    Load all .npy files and create X (features) and y (labels)
    """
    base_path = Path(base_dir)
    
    if not base_path.exists():
        print(f" Directory not found: {base_path}")
        return None, None
    
    X = []  # Features
    y = []  # Labels
    
    # Get all class folders
    class_folders = sorted([d for d in base_path.iterdir() if d.is_dir()])
    
    print(f"Found {len(class_folders)} classes")
    print("-" * 60)
    
    for class_idx, class_path in enumerate(class_folders):
        class_name = class_path.name
        npy_files = sorted(class_path.glob("*.npy"))
        
        print(f"  {class_idx}: {class_name} - {len(npy_files)} samples")
        
        for file_path in npy_files:
            try:
                data = np.load(file_path)
                
                # Ensure correct shape
                if data.shape != (SEQUENCE_LENGTH, FEATURES_PER_FRAME):
                    print(f"     Skipping {file_path.name}: shape {data.shape}")
                    continue
                
                X.append(data)
                y.append(class_idx)
                
            except Exception as e:
                print(f"     Error loading {file_path.name}: {e}")
    
    X = np.array(X)
    y = np.array(y)
    
    print("-" * 60)
    print(f" Total samples loaded: {len(X)}")
    print(f"   Shape: {X.shape}")
    print(f"   Classes: {len(np.unique(y))}")
    
    return X, y, [d.name for d in class_folders]

# Load data
X, y, class_names = load_data(ENGINEERED_DIR)

if X is None:
    print(" No data loaded!")
    exit()

Found 20 classes
------------------------------------------------------------
  0: កាតាប - 561 samples
  1: កាតាបស្ពាយក្រោយ - 482 samples
  2: កុំព្យូទ័រ - 668 samples
  3: កៅអី - 345 samples
  4: ក្ដារខៀន - 359 samples
  5: ខ្មៅដៃ - 571 samples
  6: ជ័រលុប - 400 samples
  7: ដីស - 433 samples
  8: តុ - 406 samples
  9: ទឹកលុប - 798 samples
  10: នាយករង - 430 samples
  11: នាយិកា - 627 samples
  12: បន្ទាត់ - 266 samples
  13: ប៊ិក - 583 samples
  14: ប៊ិកក្រហម - 881 samples
  15: ប៊ិកខៀវ - 962 samples
  16: សៀវភៅ - 462 samples
  17: ហ្វឺតក្រហម - 895 samples
  18: ហ្វឺតខៀវ - 862 samples
  19: ហ្វឺតខ្មៅ - 652 samples
------------------------------------------------------------
 Total samples loaded: 11643
   Shape: (11643, 30, 686)
   Classes: 20


## Step 2 Train test spilt 

In [3]:
# ============================================================
# SPLIT DATA - 80% TRAINING, 20% FOR VAL + TEST
# ============================================================

def split_data(X, y, train_size=0.8, val_ratio=0.5):
    """
    Split data into train, validation, and test sets
    - 80% for training
    - Remaining 20% split equally between val and test (10% each)
    
    Args:
        train_size: Proportion for training (default: 0.8)
        val_ratio: Proportion of remaining data for validation (default: 0.5)
                   If 0.5, val and test are equal (10% each)
                   If 0.7, val=14%, test=6%
                   If 0.3, val=6%, test=14%
    """
    # First split: train and temp (val + test)
    X_train, X_temp, y_train, y_temp = train_test_split(
        X, y, 
        train_size=train_size, 
        random_state=42, 
        stratify=y
    )
    
    # Second split: val and test from temp
    val_size = val_ratio  # Proportion of temp for validation
    X_val, X_test, y_val, y_test = train_test_split(
        X_temp, y_temp,
        test_size=(1 - val_ratio),
        random_state=42,
        stratify=y_temp
    )
    
    print("\n" + "=" * 70)
    print("DATA SPLIT SUMMARY")
    print("=" * 70)
    print(f"Total samples:  {len(X)}")
    print(f"Training:       {len(X_train):5d} ({len(X_train)/len(X)*100:5.1f}%)")
    print(f"Validation:     {len(X_val):5d} ({len(X_val)/len(X)*100:5.1f}%)")
    print(f"Test:           {len(X_test):5d} ({len(X_test)/len(X)*100:5.1f}%)")
    print("-" * 70)
    print(f"Train/Val/Test: {len(X_train)}/{len(X_val)}/{len(X_test)}")
    
    return X_train, X_val, X_test, y_train, y_val, y_test

# Use 80% training, 10% validation, 10% test
X_train, X_val, X_test, y_train, y_val, y_test = split_data(
    X, y, 
    train_size=0.8,  # 80% training
    val_ratio=0.5    # 50% of remaining for val (10% of total)
)

# One-hot encode labels
num_classes = len(class_names)
y_train_cat = to_categorical(y_train, num_classes)
y_val_cat = to_categorical(y_val, num_classes)
y_test_cat = to_categorical(y_test, num_classes)

print(f"\nClasses: {num_classes}")
print(f"Class names: {class_names}")

# Show per-class distribution in each split
print("\n" + "=" * 70)
print("PER-CLASS DISTRIBUTION IN SPLITS")
print("=" * 70)
print(f"{'Class':<20} {'Train':<8} {'Val':<8} {'Test':<8} {'Total':<8}")
print("-" * 70)

for i, class_name in enumerate(class_names):
    train_count = np.sum(y_train == i)
    val_count = np.sum(y_val == i)
    test_count = np.sum(y_test == i)
    total_count = train_count + val_count + test_count
    print(f"{class_name:<20} {train_count:<8} {val_count:<8} {test_count:<8} {total_count:<8}")



DATA SPLIT SUMMARY
Total samples:  11643
Training:        9314 ( 80.0%)
Validation:      1164 ( 10.0%)
Test:            1165 ( 10.0%)
----------------------------------------------------------------------
Train/Val/Test: 9314/1164/1165

Classes: 20
Class names: ['កាតាប', 'កាតាបស្ពាយក្រោយ', 'កុំព្យូទ័រ', 'កៅអី', 'ក្ដារខៀន', 'ខ្មៅដៃ', 'ជ័រលុប', 'ដីស', 'តុ', 'ទឹកលុប', 'នាយករង', 'នាយិកា', 'បន្ទាត់', 'ប៊ិក', 'ប៊ិកក្រហម', 'ប៊ិកខៀវ', 'សៀវភៅ', 'ហ្វឺតក្រហម', 'ហ្វឺតខៀវ', 'ហ្វឺតខ្មៅ']

PER-CLASS DISTRIBUTION IN SPLITS
Class                Train    Val      Test     Total   
----------------------------------------------------------------------
កាតាប                449      56       56       561     
កាតាបស្ពាយក្រោយ      386      48       48       482     
កុំព្យូទ័រ           534      67       67       668     
កៅអី                 276      35       34       345     
ក្ដារខៀន             287      36       36       359     
ខ្មៅដៃ               457      57       57       571     
ជ័រលុប          

## Step 3 Build Model 

### GRU 

In [4]:
import tensorflow as tf
from tensorflow.keras import layers

def build_gru_with_attention(input_shape, num_classes):
    """
    GRU model with attention to focus on the end of the sign
    """
    inputs = tf.keras.Input(shape=input_shape)
    
    x = layers.BatchNormalization()(inputs)
    x = layers.Conv1D(filters=64, kernel_size=3, padding='same', activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.2)(x)
    
    # GRU with return_sequences=True for attention
    x = layers.GRU(
        128, 
        return_sequences=True,  # ← Key for attention
        dropout=0.3,
        recurrent_dropout=0.3
    )(x)
    
    # Self-attention to focus on important frames (especially the end)
    attention = layers.Dense(1, activation='tanh')(x)
    attention = layers.Flatten()(attention)
    attention = layers.Activation('softmax')(attention)
    attention = layers.RepeatVector(128)(attention)
    attention = layers.Permute([2, 1])(attention)
    
    # Apply attention
    x = layers.Multiply()([x, attention])
    x = layers.GlobalAveragePooling1D()(x)
    
    # Dense layers
    x = layers.Dense(64, activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.3)(x)
    
    x = layers.Dense(32, activation='relu')(x)
    x = layers.Dropout(0.2)(x)
    
    outputs = layers.Dense(num_classes, activation='softmax')(x)
    
    model = tf.keras.Model(inputs=inputs, outputs=outputs)
    return model

# Build model
input_shape = (SEQUENCE_LENGTH, FEATURES_PER_FRAME)
model = build_gru_with_attention(input_shape, num_classes)

# Compile with Adam optimizer (good balance of speed and accuracy)
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()


Model: "model"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 input_1 (InputLayer)        [(None, 30, 686)]            0         []                            
                                                                                                  
 batch_normalization (Batch  (None, 30, 686)              2744      ['input_1[0][0]']             
 Normalization)                                                                                   
                                                                                                  
 conv1d (Conv1D)             (None, 30, 64)               131776    ['batch_normalization[0][0]'] 
                                                                                                  
 batch_normalization_1 (Bat  (None, 30, 64)               256       ['conv1d[0][0]']         

### 4 Model config 

In [11]:
# ============================================================
# TRAIN WITH CALLBACKS & CLASS WEIGHTS
# ============================================================

from sklearn.utils.class_weight import compute_class_weight

def get_callbacks(model_save_path):
    """
    Create callbacks for training
    """
    callbacks = [
        # Early stopping - stop if no improvement
        tf.keras.callbacks.EarlyStopping(
            monitor='val_loss',
            patience=15,
            restore_best_weights=True,
            verbose=1
        ),
        
        # Reduce learning rate on plateau
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor='val_loss',
            factor=0.5,
            patience=10,
            min_lr=1e-6,
            verbose=1
        ),
        
        # Save best model
        tf.keras.callbacks.ModelCheckpoint(
            filepath=model_save_path,
            monitor='val_accuracy',
            save_best_only=True,
            verbose=1
        )
    ]
    
    return callbacks

# ============================================================
# COMPUTE CLASS WEIGHTS (FOR IMBALANCED DATA)
# ============================================================

print("=" * 60)
print("COMPUTING CLASS WEIGHTS")
print("=" * 60)

# Calculate class weights from training data
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)

# Convert to dictionary format for Keras
class_weight_dict = {i: class_weights[i] for i in range(len(class_weights))}

# Display class weights
print(f"\n{'Class':<25} {'Samples':<10} {'Weight':<10}")
print("-" * 60)
for i, class_name in enumerate(class_names):
    samples = np.sum(y_train == i)
    weight = class_weights[i]
    print(f"{class_name:<25} {samples:<10} {weight:<10.3f}")

print("\n" + "=" * 60)
print("CLASS WEIGHT SUMMARY")
print("=" * 60)
print(f"Min weight: {min(class_weights):.3f} (majority class)")
print(f"Max weight: {max(class_weights):.3f} (minority class)")
print(f"Weight ratio: {max(class_weights)/min(class_weights):.2f}x")

# ============================================================
# TRAIN THE MODEL WITH CLASS WEIGHTS
# ============================================================

# Model save path
MODEL_SAVE_PATH = r"D:/uni/Intern-1-Project/ksl/models/gru_best_model_v2.h5"

# Get callbacks
callbacks = get_callbacks(MODEL_SAVE_PATH)

# Train the model
print("\n" + "=" * 60)
print("STARTING TRAINING WITH CLASS WEIGHTS")
print("=" * 60)
print(f"Training samples:   {len(X_train)}")
print(f"Validation samples: {len(X_val)}")
print(f"Test samples:       {len(X_test)}")
print(f"Classes:            {len(class_names)}")
print(f"Class weights:      Applied ✓")
print("-" * 60)

history = model.fit(
    X_train, y_train_cat,
    validation_data=(X_val, y_val_cat),
    epochs=100,
    batch_size=32,
    class_weight=class_weight_dict,  # ← This handles class imbalance!
    callbacks=callbacks,
    verbose=1
)


COMPUTING CLASS WEIGHTS

Class                     Samples    Weight    
------------------------------------------------------------
កាតាប                     449        1.037     
កាតាបស្ពាយក្រោយ           386        1.206     
កុំព្យូទ័រ                534        0.872     
កៅអី                      276        1.687     
ក្ដារខៀន                  287        1.623     
ខ្មៅដៃ                    457        1.019     
ជ័រលុប                    320        1.455     
ដីស                       346        1.346     
តុ                        325        1.433     
ទឹកលុប                    638        0.730     
នាយករង                    344        1.354     
នាយិកា                    502        0.928     
បន្ទាត់                   213        2.186     
ប៊ិក                      466        0.999     
ប៊ិកក្រហម                 705        0.661     
ប៊ិកខៀវ                   769        0.606     
សៀវភៅ                     370        1.259     
ហ្វឺតក្រហម                716        0.650     
ហ្

292/292 ━━━━━━━━━━━━━━━━━━━━ 7s 18ms/step - accuracy: 0.3308 - loss: 2.0883 - val_accuracy: 0.4046 - val_loss: 2.1737 - learning_rate: 0.0010
Epoch 2/100
291/292 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.6190 - loss: 0.8762
Epoch 2: val_accuracy improved from 0.40464 to 0.73368, saving model to D:/uni/Intern-1-Project/ksl/models/gru_best_model_v2.h5


292/292 ━━━━━━━━━━━━━━━━━━━━ 5s 17ms/step - accuracy: 0.6191 - loss: 0.8757 - val_accuracy: 0.7337 - val_loss: 0.6768 - learning_rate: 0.0010
Epoch 3/100
289/292 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.6770 - loss: 0.6938
Epoch 3: val_accuracy improved from 0.73368 to 0.77320, saving model to D:/uni/Intern-1-Project/ksl/models/gru_best_model_v2.h5


292/292 ━━━━━━━━━━━━━━━━━━━━ 4s 15ms/step - accuracy: 0.6772 - loss: 0.6934 - val_accuracy: 0.7732 - val_loss: 0.5465 - learning_rate: 0.0010
Epoch 4/100
288/292 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.7074 - loss: 0.6226
Epoch 4: val_accuracy improved from 0.77320 to 0.79983, saving model to D:/uni/Intern-1-Project/ksl/models/gru_best_model_v2.h5


292/292 ━━━━━━━━━━━━━━━━━━━━ 4s 15ms/step - accuracy: 0.7076 - loss: 0.6219 - val_accuracy: 0.7998 - val_loss: 0.4777 - learning_rate: 0.0010
Epoch 5/100
291/292 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.7218 - loss: 0.6173
Epoch 5: val_accuracy did not improve from 0.79983
292/292 ━━━━━━━━━━━━━━━━━━━━ 5s 16ms/step - accuracy: 0.7218 - loss: 0.6170 - val_accuracy: 0.7930 - val_loss: 0.4575 - learning_rate: 0.0010
Epoch 6/100
292/292 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.7511 - loss: 0.5183
Epoch 6: val_accuracy did not improve from 0.79983
292/292 ━━━━━━━━━━━━━━━━━━━━ 5s 16ms/step - accuracy: 0.7511 - loss: 0.5182 - val_accuracy: 0.7955 - val_loss: 0.4475 - learning_rate: 0.0010
Epoch 7/100
292/292 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.7531 - loss: 0.5100
Epoch 7: val_accuracy improved from 0.79983 to 0.82388, saving model to D:/uni/Intern-1-Project/ksl/models/gru_best_model_v2.h5


...

292/292 ━━━━━━━━━━━━━━━━━━━━ 5s 17ms/step - accuracy: 0.8790 - loss: 0.2254 - val_accuracy: 0.9141 - val_loss: 0.2509 - learning_rate: 0.0010
Epoch 59/100
291/292 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.8751 - loss: 0.2290
Epoch 59: val_accuracy did not improve from 0.91409
292/292 ━━━━━━━━━━━━━━━━━━━━ 5s 17ms/step - accuracy: 0.8752 - loss: 0.2290 - val_accuracy: 0.9081 - val_loss: 0.2356 - learning_rate: 0.0010
Epoch 60/100
289/292 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.8791 - loss: 0.2341
Epoch 60: val_accuracy improved from 0.91409 to 0.91753, saving model to D:/uni/Intern-1-Project/ksl/models/gru_best_model_v2.h5


292/292 ━━━━━━━━━━━━━━━━━━━━ 5s 17ms/step - accuracy: 0.8791 - loss: 0.2341 - val_accuracy: 0.9175 - val_loss: 0.2373 - learning_rate: 0.0010
Epoch 61/100
292/292 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.8756 - loss: 0.2457
Epoch 61: val_accuracy did not improve from 0.91753
292/292 ━━━━━━━━━━━━━━━━━━━━ 5s 16ms/step - accuracy: 0.8755 - loss: 0.2457 - val_accuracy: 0.8995 - val_loss: 0.2577 - learning_rate: 0.0010
Epoch 62/100
289/292 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.8698 - loss: 0.2472
Epoch 62: val_accuracy did not improve from 0.91753
292/292 ━━━━━━━━━━━━━━━━━━━━ 5s 15ms/step - accuracy: 0.8699 - loss: 0.2470 - val_accuracy: 0.9072 - val_loss: 0.2470 - learning_rate: 0.0010
Epoch 63/100
291/292 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.8844 - loss: 0.2140
Epoch 63: val_accuracy did not improve from 0.91753
292/292 ━━━━━━━━━━━━━━━━━━━━ 4s 15ms/step - accuracy: 0.8844 - loss: 0.2140 - val_accuracy: 0.9115 - val_loss: 0.2350 - learning_rate: 0.0010
Epoch 64/

292/292 ━━━━━━━━━━━━━━━━━━━━ 5s 16ms/step - accuracy: 0.8904 - loss: 0.2111 - val_accuracy: 0.9184 - val_loss: 0.2235 - learning_rate: 0.0010
Epoch 83/100
289/292 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.8969 - loss: 0.2072
Epoch 83: val_accuracy improved from 0.91838 to 0.92268, saving model to D:/uni/Intern-1-Project/ksl/models/gru_best_model_v2.h5


292/292 ━━━━━━━━━━━━━━━━━━━━ 5s 16ms/step - accuracy: 0.8969 - loss: 0.2071 - val_accuracy: 0.9227 - val_loss: 0.2367 - learning_rate: 0.0010
Epoch 84/100
289/292 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.8889 - loss: 0.2219
Epoch 84: val_accuracy did not improve from 0.92268
292/292 ━━━━━━━━━━━━━━━━━━━━ 5s 16ms/step - accuracy: 0.8889 - loss: 0.2217 - val_accuracy: 0.9107 - val_loss: 0.2234 - learning_rate: 0.0010
Epoch 85/100
291/292 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.8975 - loss: 0.2146
Epoch 85: val_accuracy did not improve from 0.92268
292/292 ━━━━━━━━━━━━━━━━━━━━ 5s 16ms/step - accuracy: 0.8974 - loss: 0.2146 - val_accuracy: 0.9227 - val_loss: 0.2166 - learning_rate: 0.0010
Epoch 86/100
289/292 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.8933 - loss: 0.2100
Epoch 86: val_accuracy did not improve from 0.92268
292/292 ━━━━━━━━━━━━━━━━━━━━ 5s 16ms/step - accuracy: 0.8934 - loss: 0.2099 - val_accuracy: 0.9218 - val_loss: 0.2371 - learning_rate: 0.0010
Epoch 87/

292/292 ━━━━━━━━━━━━━━━━━━━━ 5s 16ms/step - accuracy: 0.9127 - loss: 0.1707 - val_accuracy: 0.9278 - val_loss: 0.2143 - learning_rate: 5.0000e-04
Epoch 92/100
289/292 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.9149 - loss: 0.1605
Epoch 92: val_accuracy did not improve from 0.92784
292/292 ━━━━━━━━━━━━━━━━━━━━ 5s 16ms/step - accuracy: 0.9149 - loss: 0.1606 - val_accuracy: 0.9158 - val_loss: 0.2173 - learning_rate: 5.0000e-04
Epoch 93/100
292/292 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.9110 - loss: 0.1686
Epoch 93: val_accuracy did not improve from 0.92784
292/292 ━━━━━━━━━━━━━━━━━━━━ 5s 16ms/step - accuracy: 0.9110 - loss: 0.1686 - val_accuracy: 0.9201 - val_loss: 0.2179 - learning_rate: 5.0000e-04
Epoch 93: early stopping
Restoring model weights from the end of the best epoch: 78.


In [2]:
import tensorflow as tf
import pickle
import json
from pathlib import Path

# ============================================================
# LOAD SAVED MODEL (Skip Training)
# ============================================================

# Path to your saved model
MODEL_SAVE_PATH = r"D:/uni/Intern-1-Project/ksl/models/gru_best_model_v2.h5"

# Load the model
model = tf.keras.models.load_model(MODEL_SAVE_PATH)
print(f" Model loaded from: {MODEL_SAVE_PATH}")

# Verify model works
print(f" Model ready for inference!")
print(f"   Input shape: {model.input_shape}")
print(f"   Output shape: {model.output_shape}")

 Model loaded from: D:/uni/Intern-1-Project/ksl/models/gru_best_model_v2.h5
 Model ready for inference!
   Input shape: (None, 30, 686)
   Output shape: (None, 20)


## Step 5 model evaluation 

In [ ]:
%pip install seaborn

In [6]:
# ============================================================
# VISUALIZE TRAINING HISTORY
# ============================================================
import matplotlib.pyplot as plt
import seaborn as sns

def plot_training_history(history):
    """
    Plot training and validation loss/accuracy
    """
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    # Loss plot
    ax1.plot(history.history['loss'], label='Training Loss', linewidth=2)
    ax1.plot(history.history['val_loss'], label='Validation Loss', linewidth=2)
    ax1.set_title('Model Loss (With Class Weights)', fontsize=14)
    ax1.set_xlabel('Epoch', fontsize=12)
    ax1.set_ylabel('Loss', fontsize=12)
    ax1.legend(fontsize=12)
    ax1.grid(True, alpha=0.3)
    
    # Accuracy plot
    ax2.plot(history.history['accuracy'], label='Training Accuracy', linewidth=2)
    ax2.plot(history.history['val_accuracy'], label='Validation Accuracy', linewidth=2)
    ax2.set_title('Model Accuracy (With Class Weights)', fontsize=14)
    ax2.set_xlabel('Epoch', fontsize=12)
    ax2.set_ylabel('Accuracy', fontsize=12)
    ax2.legend(fontsize=12)
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(f"{Path(MODEL_SAVE_PATH).parent}/training_history.png", dpi=150)
    plt.show()

# Plot training history
plot_training_history(history)

# ============================================================
# EVALUATE ON TEST SET
# ============================================================

print("\n" + "=" * 60)
print("TEST SET EVALUATION")
print("=" * 60)

test_loss, test_accuracy = model.evaluate(X_test, y_test_cat, verbose=0)
print(f"Test Loss:     {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy:.4f} ({test_accuracy*100:.2f}%)")

# ============================================================
# PER-CLASS ACCURACY (Most important for imbalanced data)
# ============================================================

from sklearn.metrics import confusion_matrix, classification_report

y_pred = model.predict(X_test, verbose=0)
y_pred_classes = np.argmax(y_pred, axis=1)

print("\n" + "=" * 60)
print("PER-CLASS ACCURACY (With Class Weights)")
print("=" * 60)

cm = confusion_matrix(y_test, y_pred_classes)
class_accuracy = cm.diagonal() / cm.sum(axis=1)

print(f"\n{'Class':<25} {'Accuracy':<12} {'Samples':<10} {'Status':<10}")
print("-" * 65)

problematic = []
for i, class_name in enumerate(class_names):
    acc = class_accuracy[i]
    samples = np.sum(y_test == i)
    status = " Good" if acc >= 0.8 else " Okay" if acc >= 0.7 else " Poor"
    print(f"{class_name:<25} {acc*100:>6.2f}%     {samples:<10} {status:<10}")
    if acc < 0.7:
        problematic.append((class_name, acc))

# Show problematic classes
if problematic:
    print("\n" + "=" * 60)
    print(" PROBLEMATIC CLASSES (Below 70% Accuracy)")
    print("=" * 60)
    for class_name, acc in problematic:
        print(f"  - {class_name}: {acc*100:.2f}%")
    print("\n Consider:")
    print("  - Adding more samples for these classes")
    print("  - Trying data augmentation")
    print("  - Adjusting model architecture")
else:
    print("\n" + "=" * 60)
    print(" ALL CLASSES ABOVE 70% ACCURACY!")
    print("   Class weights are working well!")
    print("=" * 60)

# ============================================================
# CONFUSION MATRIX
# ============================================================

plt.figure(figsize=(16, 14))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names)
plt.title('Confusion Matrix (With Class Weights)', fontsize=16)
plt.xlabel('Predicted', fontsize=14)
plt.ylabel('Actual', fontsize=14)
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig(f"{Path(MODEL_SAVE_PATH).parent}/confusion_matrix.png", dpi=150)
plt.show()

# ============================================================
# CLASSIFICATION REPORT
# ============================================================

print("\n" + "=" * 60)
print("CLASSIFICATION REPORT")
print("=" * 60)
print(classification_report(y_test, y_pred_classes, target_names=class_names))

NameError: name 'history' is not defined

## Step 6 Save model

In [14]:
# ============================================================
# SAVE MODEL AND HISTORY
# ============================================================

import pickle
import json 
# Save the model
model.save(MODEL_SAVE_PATH)
print(f"\n Model saved to: {MODEL_SAVE_PATH}")

# Save class names
# Save as pickle (for Python)
with open(f"{Path(MODEL_SAVE_PATH).parent}/class_names.pkl", 'wb') as f:
    pickle.dump(class_names, f)
print(f" Class names saved to: class_names.pkl")

# Save as JSON (more universal, readable)
with open(f"{Path(MODEL_SAVE_PATH).parent}/label_map.json", 'w', encoding='utf-8') as f:
    json.dump({
        str(i): class_name for i, class_name in enumerate(class_names)
    }, f, ensure_ascii=False, indent=2)
print(f" Label map saved to: label_map.json")

# Save training history
with open(f"{Path(MODEL_SAVE_PATH).parent}/training_history.pkl", 'wb') as f:
    pickle.dump(history.history, f)
print(f" Training history saved to: {Path(MODEL_SAVE_PATH).parent}/training_history.pkl")

print("\n" + "=" * 60)
print(" TRAINING COMPLETE!")
print("=" * 60)


 Model saved to: D:/uni/Intern-1-Project/ksl/models/gru_best_model_v2.h5
 Class names saved to: class_names.pkl
 Label map saved to: label_map.json


AttributeError: 'dict' object has no attribute 'history'